# Father Geula — VO Alignment + Job Bundles v3.6

1. Drop the voiceover and its **ElevenLabs v3 script** into **`FatherGeula/VO/in`** with the **same filename**
   (`36 - THE F.A.T.H.E.R. METHOD.mp3` + `36 - THE F.A.T.H.E.R. METHOD.txt`). Audio tags can stay in.
2. **Runtime → Run all.** One tap on the Drive dialog, nothing else.
3. Ready-to-send zips land in **`FatherGeula/VO/bundles/`**, one per job, each with a `_kickoff.txt` beside it.

### Sending a job
New chat in **Claude** (with code execution on) or **ChatGPT** → attach `<slug>_input.zip` → paste the
message from `<slug>_kickoff.txt`. The bundle holds everything the master prompt asks for, already named
the way its Part 29 expects:

```text
<slug>_input/
├── START_HERE.md            index + alignment facts + warnings
├── bundle_manifest.json     title, slug, QC, sha256 of every file
├── MASTER_PROMPT.md         from FatherGeula/VO/master_prompt.md
├── elevenlabs_script.txt    the v3 script exactly as written, tags intact
├── voiceover.mp3
├── alignment.json           renderer handoff
├── subtitles.srt            alignment.srt, unchanged
├── voice_alignment.json     creative timing authority (new in v3.6)
└── character_sheet.jpg      from FatherGeula/VO/character_sheet.*
```

The kickoff message matters: assistants treat instructions found *inside* an uploaded file as data, so the
go-ahead has to come from your message. Edit `master_prompt.md` or swap the character sheet and every
bundle rebuilds itself on the next run. Only MISMATCH LIKELY jobs are left out.

### ElevenLabs v3 scripts
No more hand-cleaning. Before aligning, the notebook:

- **sets aside v3 audio tags** — `[pause]`, `[whispers]`, `[sighs]`, `<break time="1s"/>` — and records
  where each one sat, so `voice_alignment.json → performance_tags` tells the director that `[long pause]`
  falls between “1990s.” and “Voice” at 16.0 s;
- **removes document furniture** — markdown headings, `Notes:` / `Caption:` / `Title:` lines, `SCENE 2`, rules;
- **lets the audio decide** on anything uncertain — a title line at the top, an ALL-CAPS line, a `HOOK:` line.
  It scores the transcript with and without that line against the actual voiceover and keeps whichever
  explains the audio better. Each decision is printed and saved in `voice_alignment.json`.

The exact text that was aligned is still saved as `alignment_files/script_used.txt`.

### Script formats
`.txt` · `.md` · `.docx` · native **Google Doc** (`.gdoc`). Preference txt → md → docx → gdoc.
Google Docs on a phone: **File → Download → .docx** into `in`, or **Share → Anyone with the link** and save
the link as a one-line `.txt` named like the audio. The `.gdoc` login often fails on mobile.

### How the alignment works
Your script is ground truth and the model finds where each word lands in the audio, so **every word gets a
measured start, end and confidence**. There is **no pip install**: torchaudio's `MMS_FA` aligner ships with
Colab. Forced alignment cannot tell you the script is wrong — it will smear wrong text across the audio — so
every job gets a verdict, **OK**, **CHECK** or **MISMATCH LIKELY**.

### Outputs per job (in Drive)
```text
out/<name>/
├── alignment.json
├── alignment.srt
└── alignment_files/
    ├── script_original.txt     the script as written (v3.6)
    ├── script_used.txt         what was aligned
    ├── voice_alignment.json    sentences, pauses, words, [?] words, v3 tag positions (v3.6)
    └── voice_alignment.txt     the same, for reading
```

### Bulk
Drop as many pairs as you like into `in` and run once. Jobs that already have an `alignment.json` are not
re-aligned unless `FORCE_REDO = True`, but cell 5 still bundles them — including everything aligned with
older versions.


In [ ]:
# ═══ 1) SETUP — Drive + check the aligner is available ═══════════════════════
# No pip install. Everything used here already ships with Colab.

import os, sys, shutil, pathlib, subprocess, logging

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)  # harmless timeout chatter

PROJECT_DIR    = "FatherGeula/VO"   # folder inside your Google Drive
CACHE_IN_DRIVE = True               # keep the ~1.2 GB aligner model in Drive so it
                                    # downloads once instead of every session

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")           # one tap, once per session
    DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
except Exception as e:
    print("Drive not available (", e, ") — using local /content storage.")

BASE       = pathlib.Path("/content/drive/MyDrive") / PROJECT_DIR if DRIVE_OK else pathlib.Path("/content/VO")
IN_DIR     = BASE / "in"
OUT_DIR    = BASE / "out"
DONE_DIR   = BASE / "in_processed"
BUNDLE_DIR = BASE / "bundles"           # ready-to-send zips land here
for d in (IN_DIR, OUT_DIR, DONE_DIR, BUNDLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

if DRIVE_OK and CACHE_IN_DRIVE:
    cache = BASE / "model_cache"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(cache)   # first run downloads, later runs reuse

print("\nInput folder :", IN_DIR)
print("Output folder:", OUT_DIR)
print("Bundles      :", BUNDLE_DIR)

# ---- checks -----------------------------------------------------------------
import torch, torchaudio
print("\ntorch", torch.__version__, "| torchaudio", torchaudio.__version__,
      "| GPU:", torch.cuda.is_available())

try:
    from torchaudio.pipelines import MMS_FA
    MMS_OK = hasattr(MMS_FA, "get_aligner")
except Exception:
    MMS_OK = False

if not MMS_OK:
    print("\n⚠️  This torchaudio has no MMS_FA aligner. Run in a cell, then restart:")
    print("    !pip install -q -U torchaudio")
else:
    print("MMS_FA forced aligner available.")

FFMPEG = shutil.which("ffmpeg")
print("ffmpeg:", FFMPEG or "MISSING — run  !apt-get -qq install ffmpeg")


Mounted at /content/drive

Input folder : /content/drive/MyDrive/FatherGeula/VO/in
Output folder: /content/drive/MyDrive/FatherGeula/VO/out
Bundles      : /content/drive/MyDrive/FatherGeula/VO/bundles

torch 2.11.0+cu130 | torchaudio 2.11.0+cu130 | GPU: True
MMS_FA forced aligner available.
ffmpeg: /usr/bin/ffmpeg


In [ ]:
# ═══ 2) SETTINGS + WHAT WILL BE PROCESSED ═══════════════════════════════════
# Drop matching pairs in the "in" folder, e.g.  2026-09-06-reel.mp3 + 2026-09-06-reel.txt
# Same filename, different extension = one job. Several pairs = a batch.
# The script can be the ElevenLabs v3 version, audio tags and all.

import pathlib, datetime

if "IN_DIR" not in globals():
    raise RuntimeError("Run cell 1 first (Runtime → Run all is the easy way).")

SCORE_FLOOR     = 0.30    # per-word alignment score below this is flagged
PAUSE_THRESHOLD = 0.35    # seconds — gaps at or above this are reported as pauses
FORCE_REDO      = True   # True = re-run jobs that already have an output folder
MOVE_WHEN_DONE  = False   # True = move finished inputs into in_processed/
PREVIEW_CHARS   = 1200    # keep small; phones choke on huge cell outputs
DETAILS_DIRNAME = "alignment_files"  # script copies + voice_alignment.json/.txt

# Script cleaning — what ElevenLabs did not speak never reaches the aligner
STRIP_SQUARE_BRACKETS = True    # v3 audio tags: [pause], [whispers], [sighs], [B-ROLL: …]
STRIP_PARENS          = False   # parentheses usually contain spoken words — keep them
RESOLVE_AMBIGUOUS     = True    # titles / ALL-CAPS lines: let the audio decide if they were spoken

# Bundles — cell 5 turns every finished job into one zip you send to Claude / ChatGPT
MAKE_BUNDLES    = True
BUNDLE_MISMATCH = False   # True = also bundle MISMATCH LIKELY jobs (not recommended)
REBUILD_BUNDLES = False   # True = rebuild all; False = only new or out-of-date bundles

AUDIO_EXTS  = {".mp3", ".wav", ".m4a", ".mp4", ".flac", ".ogg", ".aac", ".opus", ".mov"}
SCRIPT_EXTS = [".txt", ".md", ".docx", ".gdoc"]   # in order of preference

# Shared files that live once in FatherGeula/VO and go into every bundle
MASTER_PROMPT_NAME   = "master_prompt"
MASTER_PROMPT_EXTS   = [".md", ".txt", ".docx", ".gdoc"]
CHARACTER_SHEET_NAME = "character_sheet"
CHARACTER_SHEET_EXTS = [".png", ".jpg", ".jpeg", ".webp"]

def find_shared(name, exts):
    """First file in the VO folder called <name>.<ext>, case-insensitive, by ext preference."""
    files = [p for p in BASE.iterdir() if p.is_file()]
    for ext in exts:
        for p in files:
            if p.stem.lower() == name and p.suffix.lower() == ext:
                return p
    return None

def find_jobs():
    audio, scripts = {}, {}
    for p in sorted(IN_DIR.iterdir()):
        if p.name.startswith(".") or p.name.startswith("~$") or not p.is_file():
            continue
        ext, stem = p.suffix.lower(), p.stem.lower()
        if ext in AUDIO_EXTS:
            audio[stem] = p
        elif ext in SCRIPT_EXTS:
            # if both .txt and .docx exist for one stem, take the preferred one
            prev = scripts.get(stem)
            if prev is None or SCRIPT_EXTS.index(ext) < SCRIPT_EXTS.index(prev.suffix.lower()):
                scripts[stem] = p

    jobs, problems = [], []
    for stem in sorted(set(audio) & set(scripts)):
        jobs.append({"stem": stem, "audio": audio[stem], "script": scripts[stem]})

    lone_a = [k for k in audio if k not in scripts]
    lone_s = [k for k in scripts if k not in audio]
    if len(lone_a) == 1 and len(lone_s) == 1:      # one lonely pair gets matched anyway
        jobs.append({"stem": lone_a[0], "audio": audio[lone_a[0]], "script": scripts[lone_s[0]]})
    else:
        problems += [f"audio with no matching script: {audio[k].name}" for k in lone_a]
        problems += [f"script with no matching audio: {scripts[k].name}" for k in lone_s]

    if not FORCE_REDO:
        kept, done = [], 0
        for j in jobs:
            if (OUT_DIR / j["stem"] / "alignment.json").exists():
                done += 1
            else:
                kept.append(j)
        if done:
            problems.append(f"{done} job(s) already aligned — skipped (FORCE_REDO = True to redo). "
                            "Cell 5 still bundles them.")
        jobs = kept
    return jobs, problems

JOBS, PROBLEMS = find_jobs()

mp = find_shared(MASTER_PROMPT_NAME, MASTER_PROMPT_EXTS)
cs = find_shared(CHARACTER_SHEET_NAME, CHARACTER_SHEET_EXTS)
print("Input folder   :", IN_DIR)
print("Master prompt  :", mp.name + " ✓" if mp else f"MISSING — add {MASTER_PROMPT_NAME}.md to {BASE}")
print("Character sheet:", cs.name + " ✓" if cs else "missing — bundles will carry a warning")
print("Scripts accepted:", ", ".join(SCRIPT_EXTS), "\n")
for p in PROBLEMS:
    print("  •", p)
if JOBS:
    print(f"\n{len(JOBS)} job(s) queued for alignment:")
    for j in JOBS:
        mb = j["audio"].stat().st_size / 1e6
        print(f"  • {j['audio'].name}  ({mb:.1f} MB)  +  {j['script'].name}")
else:
    print("\nNothing new to align. Put an audio file and a script with the SAME name in the folder above.")


Input folder   : /content/drive/MyDrive/FatherGeula/VO/in
Master prompt  : master_prompt.md ✓
Character sheet: character_sheet.jpg ✓
Scripts accepted: .txt, .md, .docx, .gdoc 


13 job(s) queued for alignment:
  • 23 - STRUCTURE STARVATION.mp3  (1.6 MB)  +  23 - STRUCTURE STARVATION.txt
  • 24 - IS FASTING JUST SELF-HARM.mp3  (1.5 MB)  +  24 - IS FASTING JUST SELF-HARM.txt
  • 25 - THE TWO SOLDIERS ON THE WALL.mp3  (1.5 MB)  +  25 - THE TWO SOLDIERS ON THE WALL.txt
  • 26 - A DAY UNDER THE RULE.mp3  (2.3 MB)  +  26 - A DAY UNDER THE RULE.txt
  • 28 - HAMMER — 1 CORINTHIANS 9-27.mp3  (0.3 MB)  +  28 - HAMMER — 1 CORINTHIANS 9-27.txt
  • 29 - CAN I PRAY IF I DON'T BELIEVE YET.mp3  (1.3 MB)  +  29 - CAN I PRAY IF I DON'T BELIEVE YET.txt
  • 30 - HOW TO FAST WITHOUT WRECKING YOURSELF.mp3  (1.4 MB)  +  30 - HOW TO FAST WITHOUT WRECKING YOURSELF.txt
  • 31 - ON BEING MADE.mp3  (1.4 MB)  +  31 - ON BEING MADE.txt
  • 32 - THE MAN COMFORT MAKES.mp3  (1.6 MB)  +  32 - THE MAN COMFORT MAKES.txt


In [ ]:
# ═══ 3) ENGINE — definitions only, nothing runs yet ══════════════════════════
import os, re, json, gc, math, datetime, subprocess, tempfile, statistics, pathlib
import hashlib, shutil, unicodedata, zipfile

NOTEBOOK_VERSION = "3.6"

# ---------- numbers → words ----------------------------------------------------
_ONES = ["zero","one","two","three","four","five","six","seven","eight","nine","ten",
         "eleven","twelve","thirteen","fourteen","fifteen","sixteen","seventeen",
         "eighteen","nineteen"]
_TENS = ["","","twenty","thirty","forty","fifty","sixty","seventy","eighty","ninety"]

def _under_thousand(n):
    out = []
    if n >= 100:
        out += [_ONES[n // 100], "hundred"]; n %= 100
    if n >= 20:
        out.append(_TENS[n // 10]); n %= 10
        if n: out.append(_ONES[n])
    elif n:
        out.append(_ONES[n])
    return out

def num_to_words(n):
    if n == 0: return ["zero"]
    out = []
    if n >= 1_000_000:
        out += _under_thousand(n // 1_000_000) + ["million"]; n %= 1_000_000
    if n >= 1000:
        # years: 1995 → nineteen ninety five; 2000–2009 stay "two thousand five"
        if 1100 <= n <= 2099 and n % 100 != 0 and not 2000 <= n <= 2009:
            return _under_thousand(n // 100) + _under_thousand(n % 100)
        out += _under_thousand(n // 1000) + ["thousand"]; n %= 1000
    if n: out += _under_thousand(n)
    return out

_ORD_IRREG = {"one": "first", "two": "second", "three": "third", "five": "fifth",
              "eight": "eighth", "nine": "ninth", "twelve": "twelfth"}

def ordinal_words(n):
    w = num_to_words(n)
    last = w[-1]
    if last in _ORD_IRREG:      w[-1] = _ORD_IRREG[last]
    elif last.endswith("y"):    w[-1] = last[:-1] + "ieth"
    else:                       w[-1] = last + "th"
    return w

def plural_number_words(n):          # 1990s → nineteen nineties
    w = num_to_words(n)
    w[-1] = w[-1][:-1] + "ies" if w[-1].endswith("y") else w[-1] + "s"
    return w

# ---------- script → aligner tokens ------------------------------------------
_APOS = {"’": "'", "ʼ": "'", "´": "'"}
WORD_RE = re.compile(r"\S+")

def script_tokens(text):
    return [{"text": m.group(0), "char_start": m.start(), "char_end": m.end()}
            for m in WORD_RE.finditer(text)]

def normalize_for_aligner(word, vocab):
    """Lowercase, expand numbers, drop anything the aligner's dictionary can't take.
    Returns "" for tokens that carry no sound (a lone dash, quote marks…).
    v3.6: '16:32', '1,000', '3.5', '4th', '1990s', '100%' are spoken, not dropped."""
    s = word.lower()
    for k, v in _APOS.items():
        s = s.replace(k, v)
    s = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", s)                          # 1,000 → 1000
    s = re.sub(r"(?<=\d)[:.](?=\d)", lambda m: " point " if m.group(0) == "." else " ", s)
    s = s.replace("%", " percent ").replace("&", " and ")
    s = re.sub(r"(\d+)(st|nd|rd|th)(?![a-z])",
               lambda m: " " + " ".join(ordinal_words(int(m.group(1)))) + " ", s)
    s = re.sub(r"(\d+0)'?s(?![a-z])",
               lambda m: " " + " ".join(plural_number_words(int(m.group(1)))) + " ", s)
    s = re.sub(r"(?<=[^\W\d_])(?=\d)|(?<=\d)(?=[^\W\d_])", " ", s)        # a1 → a 1
    s = re.sub(r"[-‐-―]", " ", s)     # hyphen is the aligner's blank token
    s = re.sub(r"[_/\\|]", " ", s)
    pieces = []
    for p in s.split():
        p_core = re.sub(r"^[^\w']+|[^\w']+$", "", p, flags=re.UNICODE)
        if not p_core:
            continue
        if p_core.isdigit():
            pieces += num_to_words(int(p_core))
        else:
            pieces.append(p_core)
    joined = "".join(pieces)                       # aligner wants one token per word
    return "".join(c for c in joined if c in vocab)

def aligner_input(text, vocab):
    toks = script_tokens(text)
    norm = [normalize_for_aligner(t["text"], vocab) for t in toks]
    spoken_idx = [i for i, w in enumerate(norm) if w]
    return toks, spoken_idx, [norm[i] for i in spoken_idx]

# ---------- script input: .txt / .md / .docx / Google Doc --------------------
import xml.etree.ElementTree as ET

_W = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"

def read_docx(path, skip_tables=True):
    """Plain text from a .docx using only the standard library.
    Heading paragraphs come back prefixed with '# ' so the cleaner drops them;
    table cells are skipped by default (production docs keep notes in tables)."""
    with zipfile.ZipFile(path) as z:
        root = ET.fromstring(z.read("word/document.xml"))
    parent = {c: p for p in root.iter() for c in p}

    def ancestors(el):
        while el is not None:
            el = parent.get(el)
            if el is not None:
                yield el

    paras, skipped_tables = [], 0
    for p in root.iter(_W + "p"):
        anc_tags = {a.tag for a in ancestors(p)}
        if _W + "p" in anc_tags:                 # paragraph nested in a paragraph
            continue
        if _W + "tbl" in anc_tags:
            skipped_tables += 1
            if skip_tables:
                continue
        buf = []
        for node in p.iter():
            if node.tag == _W + "t":
                buf.append(node.text or "")
            elif node.tag == _W + "tab":
                buf.append(" ")
            elif node.tag == _W + "br":
                buf.append("\n")
        text = "".join(buf)
        if not text.strip():
            continue
        style = ""
        for st in p.iter(_W + "pStyle"):
            style = (st.get(_W + "val") or "").lower()
            break
        if style.startswith("heading") or style.startswith("title") or style.startswith("subtitle"):
            text = "# " + text.strip()
        paras.append(text)

    notes = [f"skipped {skipped_tables} table paragraph(s)"] if (skip_tables and skipped_tables) else []
    return "\n\n".join(paras), notes

_DRIVE_SVC = {}

def _drive_service():
    """Drive API access. On mobile this often fails with 'credential propagation was
    unsuccessful' — the auth popup cannot complete. Callers must handle that."""
    if "svc" not in _DRIVE_SVC:
        from google.colab import auth
        auth.authenticate_user()                     # one extra tap, first time only
        from googleapiclient.discovery import build
        _DRIVE_SVC["svc"] = build("drive", "v3")
    return _DRIVE_SVC["svc"]

GDOC_HELP = (
    "Google Docs need an API login that frequently fails on phones. Two ways round it, "
    "both zero-auth:\n"
    "   1. In Docs: File → Download → Microsoft Word (.docx), save it to the in folder.\n"
    "   2. In Docs: Share → General access → Anyone with the link, copy the link, and save it "
    "as a one-line .txt named like the audio. This notebook will fetch the text from that link."
)

_DOC_URL_RE = re.compile(r"https?://docs\.google\.com/document/d/([A-Za-z0-9_-]{20,})")

def read_doc_link(text):
    """Fetch a link-shared Google Doc as plain text. No login, no API client."""
    import urllib.request, urllib.error
    m = _DOC_URL_RE.search(text)
    if not m:
        return None
    url = f"https://docs.google.com/document/d/{m.group(1)}/export?format=txt"
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            body = r.read()
    except urllib.error.HTTPError as e:
        raise ValueError(f"the Doc link returned HTTP {e.code}. Set Share → Anyone with the link.")
    out = body.decode("utf-8", "replace").lstrip("﻿")
    if out.lstrip()[:15].lower().startswith(("<!doctype", "<html")):
        raise ValueError("that Doc is not link-shared, so Google returned a sign-in page instead "
                         "of the text. Set Share → Anyone with the link, or use .docx.")
    return out

def _q_escape(name):
    return name.replace("\\", "\\\\").replace("'", "\\'")

def _drive_folder_id(path):
    """Walk the mounted path (…/MyDrive/A/B/C/x.gdoc) to the Drive id of folder C."""
    parts = list(pathlib.Path(path).parent.parts)
    anchor = next((i for i, p in enumerate(parts) if p in ("MyDrive", "My Drive")), None)
    if anchor is None:
        return None
    svc, parent = _drive_service(), "root"
    for name in parts[anchor + 1:]:
        q = (f"name = '{_q_escape(name)}' and '{parent}' in parents and "
             "mimeType = 'application/vnd.google-apps.folder' and trashed = false")
        found = svc.files().list(q=q, fields="files(id)", pageSize=2).execute().get("files", [])
        if not found:
            return None
        parent = found[0]["id"]
    return parent

def read_gdoc(path):
    """A native Google Doc has no readable bytes in the FUSE mount — the .gdoc entry
    is a stub, and opening it raises OSError 95. So look the document up by name
    through the Drive API and export it as plain text."""
    p = pathlib.Path(path)
    try:
        svc = _drive_service()
    except Exception as e:
        raise ValueError(f"could not sign in to Drive ({type(e).__name__}: {e}).\n{GDOC_HELP}")

    q = (f"name = '{_q_escape(p.stem)}' and "
         "mimeType = 'application/vnd.google-apps.document' and trashed = false")
    folder = _drive_folder_id(p)
    if folder:
        q += f" and '{folder}' in parents"

    hits = svc.files().list(q=q, fields="files(id, name, modifiedTime)",
                            orderBy="modifiedTime desc", pageSize=10
                            ).execute().get("files", [])

    if not hits and folder:      # name mismatch inside the folder — take any doc in it
        q2 = ("mimeType = 'application/vnd.google-apps.document' and trashed = false "
              f"and '{folder}' in parents")
        candidates = svc.files().list(q=q2, fields="files(id, name)", pageSize=25
                                      ).execute().get("files", [])
        hits = [c for c in candidates if c["name"].lower() == p.stem.lower()]

    if not hits:
        raise ValueError(f"no Google Doc named '{p.stem}' found in that folder.\n{GDOC_HELP}")

    data = svc.files().export(fileId=hits[0]["id"], mimeType="text/plain").execute()
    text = data.decode("utf-8") if isinstance(data, (bytes, bytearray)) else str(data)
    return text.lstrip("﻿")

def read_script_file(path):
    ext = pathlib.Path(path).suffix.lower()
    if ext == ".docx":
        text, notes = read_docx(path)
        return text, "docx", notes
    if ext == ".gdoc":
        return read_gdoc(path), "google doc", ["exported from Google Docs as plain text"]
    if ext == ".doc":
        raise ValueError("legacy .doc is not supported — save it as .docx or .txt")

    text = pathlib.Path(path).read_text(encoding="utf-8-sig")
    if len(text) < 500 and "docs.google.com/document" in text:
        linked = read_doc_link(text)          # the .txt holds a Doc link, not the script
        if linked:
            return linked, "google doc via link", ["fetched from the shared Doc link"]
    return text, ext.lstrip("."), []

# ---------- cleaning: what ElevenLabs spoke, not what the document contains ---
# Every line ends up as one of three things:
#   removed   — certainly never spoken (markdown heading, Notes:, Caption:, SCENE 2, rule)
#   spoken    — narration
#   ambiguous — a title or ALL-CAPS line that may or may not have been read aloud.
#               The audio decides (see resolve_ambiguous).
# v3 audio tags ([pause], [whispers], <break …/>) are taken out but their positions are
# kept, so voice_alignment.json can say where each performance direction sits.

_LABEL_RE   = re.compile(r"^[^a-z]*[A-Z][A-Z0-9 ,'&/-]{1,38}:?\s*$")   # HOOK:, SCENE 2, TITLE
_CERTAIN_RE = re.compile(r"^\W*(hook|cta|vo|v\.?o\.?|voice ?over|script|title|intro|outro|"
                         r"notes?|b-?roll|scene\s*\d+|shot\s*\d+|section\s*\d+)\W*$", re.I)
_META_RE    = re.compile(r"^\s*(title|notes?|scene|shot|visuals?|b-?roll|music|sfx|duration|"
                         r"length|date|version|caption|hashtags?|description|thumbnail|cover|"
                         r"on-?screen(?: text)?|tags?)\s*\d*\s*[:\-–—]\s*.*$", re.I)
# speaker / section labels need a colon — "Voice — the one inside you…" is narration
_SPOKEN_RE  = re.compile(r"^\s*(vo|v\.o\.|voice ?over|voice|narration|narrator|script|spoken)"
                         r"\s*:\s*(.*)$", re.I)
_MAYBE_RE   = re.compile(r"^\s*(hook|cta|intro|outro|opening|closing)\s*:\s*(.*)$", re.I)
_MD_HEAD_RE = re.compile(r"^\s{0,3}#{1,6}\s")
_RULE_RE    = re.compile(r"^\s*([-*_=]\s*){3,}$")
_SENT_RE    = re.compile("(\\d+)")
_TITLE_NUM  = re.compile(r"^\s*\d+\s*[-–—.)]\s*")

def _letters(s):
    return re.sub(r"[^a-z0-9]", "", unicodedata.normalize("NFKD", s).lower())

def clean_script(raw, title=None, strip_square=True, strip_parens=False):
    """Returns {"lines": [...], "tags": [...], "removals": [...]}.
    Each line: {"text", "status": spoken|ambiguous, "reason", "default_drop"}.
    Tag positions are sentinel tokens inside the line text until assemble_script()."""
    lines, tags, removals = [], [], []
    title_key = _letters(_TITLE_NUM.sub("", title)) if title else None

    def tagify(text):
        def make(kind_default):
            def f(m):
                tag = m.group(0).strip()
                kind = kind_default
                if kind_default == "audio_tag" and ":" in tag:
                    kind = "direction"                     # [B-ROLL: candles]
                tags.append({"tag": tag, "kind": kind})
                return f" {len(tags) - 1} "
            return f
        text = re.sub(r"<break\b[^>\n]*>", make("break"), text, flags=re.I)
        text = re.sub(r"<[^<>\n]{1,80}>", make("markup"), text)
        if strip_square:
            text = re.sub(r"\[[^\]\n]{0,80}\]", make("audio_tag"), text)
        if strip_parens:
            text = re.sub(r"\([^)\n]{0,80}\)", make("parenthetical"), text)
        return text

    seen_content = False
    for ln in raw.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        stripped = ln.strip()
        if not stripped:
            lines.append({"text": "", "status": "spoken", "reason": None, "default_drop": False})
            continue

        status, reason, default_drop, body = "spoken", None, False, ln
        if _MD_HEAD_RE.match(ln):
            removals.append(("markdown heading", stripped)); continue
        if _RULE_RE.match(ln):
            removals.append(("horizontal rule", stripped)); continue
        m = _SPOKEN_RE.match(ln)
        if m:
            body = m.group(2)                                # "VO: text" → text
            removals.append(("speaker label", m.group(1) + ":"))
        else:
            m = _MAYBE_RE.match(ln)
            if m:
                body = m.group(2)                            # "HOOK: text" → maybe spoken
                status, reason = "ambiguous", f"labelled “{m.group(1)}:” line"
            elif _META_RE.match(ln):
                removals.append(("document label", stripped)); continue

        body = tagify(body)
        body = re.sub(r"[*_`]{1,3}", "", body)                     # markdown emphasis
        body = re.sub(r"^\s*[-•*]\s+", "", body)                   # list bullet
        body = re.sub(r"\s+", " ", body).strip()
        plain = _SENT_RE.sub(" ", body).strip()

        if plain and status == "spoken":
            if _LABEL_RE.match(plain) and len(plain.split()) <= 6:
                if _CERTAIN_RE.match(plain) or plain.endswith(":"):
                    removals.append(("section heading", plain))
                    body = " ".join(t for t in body.split() if _SENT_RE.fullmatch(t))  # keep tags only
                    plain = ""
                else:
                    status, reason, default_drop = "ambiguous", "ALL-CAPS line (title or heading?)", True
            elif title_key and _letters(plain) == title_key:
                status, reason = "ambiguous", "same words as the filename title"
            elif (not seen_content and len(plain.split()) <= 8
                  and not re.search(r"[.!?…\"”»)]\s*$", plain)):
                status, reason = "ambiguous", "first line without punctuation (title?)"

        if plain:
            seen_content = True
        lines.append({"text": body, "status": status, "reason": reason,
                      "default_drop": default_drop})

    return {"lines": lines, "tags": tags, "removals": removals}

def assemble_script(clean, drop=frozenset()):
    """Build the narration text with `drop` line numbers removed.
    Returns (text, tag_positions). tag_positions[i]["words_before"] = how many
    script tokens precede the tag in the final text."""
    out_lines, positions, count = [], [], 0
    for i, ln in enumerate(clean["lines"]):
        real = []
        for t in ln["text"].split():
            m = _SENT_RE.fullmatch(t)
            if m:
                positions.append({**clean["tags"][int(m.group(1))], "words_before": count})
            elif i not in drop:
                real.append(t)
                count += 1
        out_lines.append(" ".join(real))
    text = "\n".join(out_lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, positions

def ambiguous_lines(clean):
    return [i for i, ln in enumerate(clean["lines"])
            if ln["status"] == "ambiguous" and _SENT_RE.sub(" ", ln["text"]).strip()]

def default_drops(clean):
    return {i for i in ambiguous_lines(clean) if clean["lines"][i]["default_drop"]}

# ---------- audio ------------------------------------------------------------
def load_audio_16k(path):
    """Decode anything (mp3/m4a/mp4/mov/…) to 16 kHz mono via ffmpeg.
    Avoids every torchaudio backend problem — ffmpeg is already in Colab."""
    import numpy as np, torch
    from scipy.io import wavfile
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        wav_path = tmp.name
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(path),
           "-ac", "1", "-ar", "16000", "-f", "wav", wav_path]
    subprocess.run(cmd, check=True)
    sr, data = wavfile.read(wav_path)
    os.remove(wav_path)
    if data.dtype.kind == "i":
        data = data.astype("float32") / float(np.iinfo(data.dtype).max)
    elif data.dtype.kind == "u":
        data = (data.astype("float32") - 128.0) / 128.0
    else:
        data = data.astype("float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    return torch.from_numpy(data).unsqueeze(0), sr

# ---------- forced alignment --------------------------------------------------
_STATE = {}

def _load_aligner():
    import torch
    from torchaudio.pipelines import MMS_FA as bundle
    if "aligner" not in _STATE:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = bundle.get_model().to(device).eval()
        d = bundle.get_dict()
        # index 0 is CTC blank (it is the "-" character here) and <star> is multi-char;
        # feeding either to the tokenizer raises "targets shouldn't contain blank index"
        allowed = {c for c, i in d.items() if i != 0 and len(c) == 1}
        _STATE.update(device=device, model=model,
                      tokenizer=bundle.get_tokenizer(),
                      aligner=bundle.get_aligner(),
                      vocab=allowed)
    return _STATE

def compute_emission(waveform):
    """Run the acoustic model once per job; every alignment attempt reuses it."""
    import torch
    st = _load_aligner()
    with torch.inference_mode():
        emission, _ = st["model"](waveform.to(st["device"]))
    return emission

def release_emission():
    import torch
    gc.collect()
    if _STATE.get("device") == "cuda":
        torch.cuda.empty_cache()

def align_words(emission, norm_words):
    """Returns (word_spans, num_frames). One span-list per entry of norm_words."""
    import torch
    st = _load_aligner()
    with torch.inference_mode():
        spans = st["aligner"](emission[0], st["tokenizer"](norm_words))
    return spans, emission.size(1)

def to_log_probs(emission):
    import torch
    return torch.log_softmax(emission[0].float(), dim=-1).cpu()

def transcript_loglik(log_probs, norm_words):
    """How well does this transcript explain the whole audio? CTC log-likelihood over
    every frame. A line that was never spoken costs a lot; a spoken line left out
    costs even more. Higher is better."""
    import torch
    import torch.nn.functional as F
    st = _load_aligner()
    ids = [i for w in st["tokenizer"](norm_words) for i in w]
    T = log_probs.shape[0]
    if not ids or len(ids) > T:
        return float("-inf")
    loss = F.ctc_loss(log_probs.unsqueeze(1),
                      torch.tensor([ids], dtype=torch.long),
                      torch.tensor([T], dtype=torch.long),
                      torch.tensor([len(ids)], dtype=torch.long),
                      blank=0, reduction="sum", zero_infinity=False)
    v = -float(loss)
    return v if math.isfinite(v) else float("-inf")

LL_MARGIN     = 1.0     # nats — dropping a line must beat keeping it by this much
MAX_AMBIGUOUS = 12      # beyond this, ambiguous lines fall back to the simple rule

def resolve_ambiguous(clean, log_probs, vocab):
    """Greedy keep/drop test per ambiguous line, judged by the audio itself.
    Returns (drop_set, decisions)."""
    amb = ambiguous_lines(clean)
    if not amb:
        return set(), []
    if not RESOLVE_AMBIGUOUS:
        drop = default_drops(clean)
        return drop, [{"line": _SENT_RE.sub(" ", clean["lines"][i]["text"]).strip(),
                       "reason": clean["lines"][i]["reason"],
                       "decision": "dropped (rule)" if i in drop else "kept (rule)"} for i in amb]

    drop, decisions = set(), []
    base = transcript_loglik(log_probs, aligner_input(assemble_script(clean, drop)[0], vocab)[2])
    tested = amb[:MAX_AMBIGUOUS]
    for i in tested:
        trial = drop | {i}
        ll = transcript_loglik(log_probs, aligner_input(assemble_script(clean, trial)[0], vocab)[2])
        gain = ll - base
        text = _SENT_RE.sub(" ", clean["lines"][i]["text"]).strip()
        if gain > LL_MARGIN:
            drop, base = trial, ll
            decisions.append({"line": text, "reason": clean["lines"][i]["reason"],
                              "decision": "dropped — not in the audio",
                              "loglik_gain": round(gain, 1)})
        else:
            decisions.append({"line": text, "reason": clean["lines"][i]["reason"],
                              "decision": "kept — spoken in the audio",
                              "loglik_gain": round(gain, 1)})
    for i in amb[MAX_AMBIGUOUS:]:
        if clean["lines"][i]["default_drop"]:
            drop.add(i)
        decisions.append({"line": _SENT_RE.sub(" ", clean["lines"][i]["text"]).strip(),
                          "reason": clean["lines"][i]["reason"],
                          "decision": "dropped (rule)" if i in drop else "kept (rule)"})
    return drop, decisions

def spans_to_times(spans, num_frames, num_samples, sample_rate=16000):
    """Frame indices → seconds, plus a length-weighted confidence per word."""
    ratio = num_samples / num_frames
    out = []
    for word_spans in spans:
        start = word_spans[0].start * ratio / sample_rate
        end   = word_spans[-1].end * ratio / sample_rate
        weight = sum(len(s) for s in word_spans) or 1
        score = sum(s.score * len(s) for s in word_spans) / weight
        out.append({"start": float(start), "end": float(end), "score": float(score)})
    return out

def build_words(script_text, timings, spoken_idx, total_tokens_source):
    """Stitch measured timings back onto every token of the approved script,
    including the punctuation-only ones that were never sent to the aligner."""
    toks = total_tokens_source
    mapped = [None] * len(toks)
    for slot, t in zip(spoken_idx, timings):
        mapped[slot] = {"start": t["start"], "end": t["end"], "score": t["score"],
                        "timing_source": "forced"}

    for i, m in enumerate(mapped):
        if m is not None:
            continue
        left  = next((k for k in range(i - 1, -1, -1) if mapped[k] is not None), None)
        right = next((k for k in range(i + 1, len(mapped)) if mapped[k] is not None), None)
        if left is not None and right is not None:
            st, en = mapped[left]["end"], mapped[right]["start"]
        elif left is not None:
            st = en = mapped[left]["end"]
        elif right is not None:
            st = en = mapped[right]["start"]
        else:
            st = en = 0.0
        mapped[i] = {"start": st, "end": max(en, st), "score": None,
                     "timing_source": "punctuation"}

    prev_end = 0.0
    for m in mapped:
        m["start"] = max(float(m["start"]), 0.0)
        if m["start"] < prev_end - 1e-6:
            m["start"] = prev_end
        m["end"] = max(float(m["end"]), m["start"])
        prev_end = m["end"]

    words = []
    for idx, (tok, m) in enumerate(zip(toks, mapped), start=1):
        words.append({"index": idx, "text": tok["text"],
                      "char_start": tok["char_start"], "char_end": tok["char_end"],
                      "start": round(m["start"], 3), "end": round(m["end"], 3),
                      "timing_source": m["timing_source"],
                      "score": None if m["score"] is None else round(m["score"], 4)})
    return words

# ---------- free QC: no second model, just the alignment's own numbers --------
def quality_check(words, audio_seconds):
    spoken = [w for w in words if w["timing_source"] == "forced"]
    scores = [w["score"] for w in spoken]
    flags, suspects = [], []

    if not spoken:
        return {"verdict": "FAILED", "flags": ["no words aligned"], "suspects": [],
                "mean_score": 0.0, "low_count": 0, "worst_window": None}

    mean_score = statistics.fmean(scores)
    low = [w for w in spoken if w["score"] < SCORE_FLOOR]

    # syllable-free duration sanity: seconds per character, vs this VO's own median
    per_char = [(w, (w["end"] - w["start"]) / max(len(w["text"]), 1)) for w in spoken]
    med = statistics.median([v for _, v in per_char])
    stretched = [w for w, v in per_char if v > med * 4 and (w["end"] - w["start"]) > 0.9]
    crushed   = [w for w, v in per_char if v < med / 4 and len(w["text"]) > 3]

    # worst 5-word run — a mismatch shows up as a cluster, not one bad word
    worst = None
    if len(spoken) >= 5:
        runs = [(statistics.fmean([x["score"] for x in spoken[i:i + 5]]), i)
                for i in range(len(spoken) - 4)]
        s, i = min(runs)
        if s < SCORE_FLOOR * 1.5:
            worst = {"mean_score": round(s, 3),
                     "text": " ".join(x["text"] for x in spoken[i:i + 5]),
                     "start": spoken[i]["start"], "end": spoken[i + 4]["end"]}

    suspects = {w["index"]: w for w in (low + stretched + crushed)}
    suspects = [suspects[k] for k in sorted(suspects)]

    if mean_score < 0.55:
        flags.append(f"low mean alignment score ({mean_score:.2f}) — the script may not match the audio")
    if len(low) > max(3, 0.05 * len(spoken)):
        flags.append(f"{len(low)} words scored under {SCORE_FLOOR} — check the script wording")
    if worst:
        flags.append(f"weak run at {worst['start']:.1f}s: “{worst['text']}”")
    if stretched:
        flags.append(f"{len(stretched)} word(s) stretched over an implausible span")
    tail = audio_seconds - spoken[-1]["end"]
    if tail > 1.5:
        flags.append(f"{tail:.1f}s of audio after the last word — is a line missing from the script?")
    if spoken[0]["start"] > 1.5:
        flags.append(f"{spoken[0]['start']:.1f}s before the first word — is a line missing from the script?")

    verdict = "OK" if not flags else ("CHECK" if mean_score >= 0.55 else "MISMATCH LIKELY")
    return {"verdict": verdict, "flags": flags, "suspects": suspects,
            "mean_score": round(mean_score, 4), "low_count": len(low), "worst_window": worst}

# ---------- grouping + formatting --------------------------------------------
SENT_END = re.compile(r"[.!?…][\"'”’»)\]]*$")

def sentences_from_words(words):
    groups, cur = [], []
    for w in words:
        cur.append(w)
        if SENT_END.search(w["text"]):
            groups.append(cur); cur = []
    if cur: groups.append(cur)
    return [{"index": n, "text": " ".join(g["text"] for g in grp),
             "start": grp[0]["start"], "end": grp[-1]["end"],
             "first_word_index": grp[0]["index"], "last_word_index": grp[-1]["index"]}
            for n, grp in enumerate(groups, start=1)]

def find_pauses(words, threshold):
    out = []
    for a, b in zip(words, words[1:]):
        gap = b["start"] - a["end"]
        if gap >= threshold:
            out.append({"start": round(a["end"], 3), "end": round(b["start"], 3),
                        "duration": round(gap, 3),
                        "after_word_index": a["index"], "before_word_index": b["index"],
                        "after_word": a["text"], "before_word": b["text"]})
    return out

def tc(seconds):
    seconds = max(0.0, float(seconds))
    return f"{int(seconds // 60):02d}:{seconds - int(seconds // 60) * 60:06.3f}"

def srt_tc(seconds):
    seconds = max(0.0, float(seconds))
    h, rem = divmod(seconds, 3600); m, s = divmod(rem, 60)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{int(round((s - int(s)) * 1000)):03d}"

# ---------- voice_alignment.json — the creative timing authority -------------
def place_tags(tag_positions, words):
    """Turn 'words_before' counts into word indexes and a time."""
    out = []
    n = len(words)
    for t in tag_positions or []:
        wb = t["words_before"]
        after  = wb if wb >= 1 else None
        before = wb + 1 if wb + 1 <= n else None
        if after:
            at = words[after - 1]["end"]
        elif before:
            at = words[before - 1]["start"]
        else:
            at = 0.0
        out.append({"tag": t["tag"], "kind": t["kind"], "at_sec": round(at, 3),
                    "after_word_index": after,
                    "after_word": words[after - 1]["text"] if after else None,
                    "before_word_index": before,
                    "before_word": words[before - 1]["text"] if before else None})
    return out

def build_voice_alignment(payload, title, tags=None, cleaning=None, tags_known=True):
    words, sents, pauses = payload["words"], payload["sentences"], payload["pauses"]
    q = payload["quality"]
    suspects = set(q.get("suspect_word_indexes", []))
    dur = payload["audio_duration_sec"]
    s0, s1 = payload.get("speech_start_sec") or 0.0, payload.get("speech_end_sec") or dur

    tag_list = place_tags(tags, words) if tags else []
    tags_after = {}
    for t in tag_list:
        if t["after_word_index"]:
            tags_after.setdefault(t["after_word_index"], []).append(t["tag"])

    return {
        "schema": "father_geula_voice_alignment_v1",
        "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "title": title,
        "voiceover_file": "voiceover.mp3",
        "source_audio_file": payload.get("audio_file"),
        "alignment_method": payload.get("alignment_method"),
        "timing_markers": {"VO_AUDIO_START": 0.0, "VO_SPEECH_START": s0,
                           "VO_SPEECH_END": s1, "VO_AUDIO_END": dur},
        "audio_duration_sec": dur,
        "leading_silence_sec": round(s0, 3),
        "trailing_silence_sec": round(dur - s1, 3),
        "quality": {"verdict": q["verdict"], "mean_score": q["mean_score"],
                    "score_floor": q.get("score_floor"), "flags": q.get("flags", [])},
        "how_to_read": [
            "All times are seconds into voiceover.mp3. Words, sentences and pauses come from "
            "the same forced alignment as alignment.json, so phrase anchors resolve identically.",
            "low_confidence: true marks the [?] words — never use one alone as a cut, text or "
            "SFX trigger.",
            "performance_tags are the ElevenLabs v3 directions from the script, placed between "
            "the words they sat between. They were not spoken.",
        ],
        "low_confidence_words": [{"index": w["index"], "text": w["text"], "start": w["start"],
                                  "end": w["end"], "score": w["score"]}
                                 for w in words if w["index"] in suspects],
        "performance_tags": tag_list,
        "performance_tags_known": tags_known,
        "script_cleaning": cleaning or {},
        "sentences": [{**s, "duration": round(s["end"] - s["start"], 3)} for s in sents],
        "pauses": [{**p, "performance_tags": tags_after.get(p["after_word_index"], [])}
                   for p in pauses],
        "words": [{"index": w["index"], "text": w["text"], "start": w["start"], "end": w["end"],
                   "score": w["score"], "timing_source": w["timing_source"],
                   "low_confidence": w["index"] in suspects} for w in words],
    }

# ---------- outputs -----------------------------------------------------------
def write_outputs(job, words, qc, audio_seconds, device, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    details_dir = out_dir / DETAILS_DIRNAME
    details_dir.mkdir(parents=True, exist_ok=True)

    sents  = sentences_from_words(words)
    pauses = find_pauses(words, PAUSE_THRESHOLD)
    spoken = [w for w in words if w["timing_source"] == "forced"]

    # alignment.json — renderer handoff, structure unchanged since v3.0
    payload = {
        "schema_version": "3.0",
        "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "audio_file": job["audio"].name,
        "script_file": job["script"].name,
        "audio_duration_sec": round(audio_seconds, 3),
        "speech_start_sec": spoken[0]["start"] if spoken else None,
        "speech_end_sec": spoken[-1]["end"] if spoken else None,
        "language": "en",
        "device": device,
        "alignment_method": "torchaudio MMS_FA CTC forced alignment against the approved script",
        "quality": {"verdict": qc["verdict"], "mean_score": qc["mean_score"],
                    "words_below_floor": qc["low_count"], "score_floor": SCORE_FLOOR,
                    "flags": qc["flags"],
                    "suspect_word_indexes": [w["index"] for w in qc["suspects"]]},
        "words": words, "sentences": sents, "pauses": pauses,
    }
    (out_dir / "alignment.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    cleaning = {"script_kind": job.get("script_kind_v3"),
                "removed_non_spoken": [{"why": w, "text": t} for w, t in job.get("script_removals", [])],
                "ambiguous_line_decisions": job.get("ambiguous_decisions", [])}
    va = build_voice_alignment(payload, job["audio"].stem, job.get("tag_positions"), cleaning)
    (details_dir / "voice_alignment.json").write_text(
        json.dumps(va, ensure_ascii=False, indent=2), encoding="utf-8")

    # Human-readable alignment
    tags_after = {}
    for t in va["performance_tags"]:
        tags_after.setdefault(t["after_word_index"] or 0, []).append(t["tag"])
    L = []
    L.append("FINAL VO ALIGNMENT — APPROVED SCRIPT WORDING")
    L.append("=" * 60)
    L.append(f"Audio: {job['audio'].name}   duration {tc(audio_seconds)}")
    L.append(f"Speech runs {tc(payload['speech_start_sec'] or 0)} → {tc(payload['speech_end_sec'] or 0)}")
    L.append(f"Every word measured by forced alignment · mean confidence {qc['mean_score']:.2f} · {qc['verdict']}")
    L.append("")
    L.append("HOW TO USE THIS")
    L.append("- Every script word has a measured start and end. These timings are the edit authority.")
    L.append("- [?] marks a word whose alignment confidence is low — verify before using it as a cut trigger.")
    L.append("- ⟨…⟩ marks an ElevenLabs v3 tag from the script. Not spoken; shown where it sat.")
    L.append("")
    if job.get("ambiguous_decisions"):
        L.append("SCRIPT LINES CHECKED AGAINST THE AUDIO")
        L.append("-" * 60)
        for d in job["ambiguous_decisions"]:
            L.append(f"{d['decision']:<30} “{d['line']}”  ({d['reason']})")
        L.append("")
    L.append("SENTENCE MAP")
    L.append("-" * 60)
    for s in sents:
        L.append(f"[{tc(s['start'])}–{tc(s['end'])}] ({s['end'] - s['start']:.2f}s) {s['text']}")
    L.append("")
    L.append(f"PAUSES ≥ {PAUSE_THRESHOLD:.2f}s  (breath points / b-roll windows)")
    L.append("-" * 60)
    L.extend([f"[{tc(p['start'])}–{tc(p['end'])}] {p['duration']:.2f}s  after “{p['after_word']}”"
              + (f"  ⟨{' '.join(p['performance_tags'])}⟩" if p["performance_tags"] else "")
              for p in va["pauses"]] or ["none"])
    L.append("")
    L.append("WORD TIMELINE")
    L.append("-" * 60)
    pause_after = {p["after_word_index"]: p for p in pauses}
    suspect_ix = {w["index"] for w in qc["suspects"]}
    for t in tags_after.get(0, []):
        L.append(f"    ⟨{t}⟩")
    for w in words:
        flag = " [?]" if w["index"] in suspect_ix else ""
        L.append(f"[{tc(w['start'])}–{tc(w['end'])}] {w['text']}{flag}")
        for t in tags_after.get(w["index"], []):
            L.append(f"    ⟨{t}⟩")
        if w["index"] in pause_after:
            p = pause_after[w["index"]]
            L.append(f"    [PAUSE {p['duration']:.3f}s — {tc(p['start'])}–{tc(p['end'])}]")
    L.append("")
    L.append("APPROVED SCRIPT (spoken wording, tags removed)")
    L.append("-" * 60)
    L.append(job["script_text"])
    (details_dir / "voice_alignment.txt").write_text("\n".join(L), encoding="utf-8")

    srt = []
    for s in sents:
        srt += [str(s["index"]), f"{srt_tc(s['start'])} --> {srt_tc(s['end'])}", s["text"], ""]
    (out_dir / "alignment.srt").write_text("\n".join(srt), encoding="utf-8")

    return {"verdict": qc["verdict"], "mean_score": qc["mean_score"],
            "words": len(spoken), "sentences": len(sents), "pauses": len(pauses),
            "suspects": len(qc["suspects"]), "tags": len(va["performance_tags"])}

# ---------- bundles: everything the master prompt needs, in one zip ----------
def slugify(title):
    s = unicodedata.normalize("NFKD", title).encode("ascii", "ignore").decode()
    s = s.lower().replace("'", "").replace(".", "")
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s or "job"

def _sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def find_input_file(name, exts):
    """Original audio/script by exact name in in/ or in_processed/, else by stem."""
    for d in (IN_DIR, DONE_DIR):
        p = d / name
        if p.is_file():
            return p
    stem = pathlib.Path(name).stem.lower()
    for d in (IN_DIR, DONE_DIR):
        for p in sorted(d.iterdir()):
            if p.is_file() and p.stem.lower() == stem and p.suffix.lower() in exts:
                return p
    return None

_TAG_HINT_RE = re.compile(r"\[[^\]\n]{1,80}\]|<break\b", re.I)

def kickoff_message(title, slug, zip_name):
    return (f"Father Geula production job: “{title}”.\n"
            f"The attached {zip_name} is a complete input bundle. Unzip it, read START_HERE.md, "
            f"then carry out MASTER_PROMPT.md exactly, using the bundle's files as the inputs it "
            f"lists. Deliver {slug}_flow_job.zip.")

def start_here_md(title, slug, files, payload, va, script_info, warnings):
    q = payload["quality"]
    low = len(va["low_confidence_words"])
    rows = [("Finished ElevenLabs v3-scored script", files["elevenlabs_script"]),
            ("Final voiceover audio", files["voiceover"]),
            ("voice_alignment.json — creative timing authority", files["voice_alignment"]),
            ("alignment.json — renderer handoff", files["alignment"]),
            ("Subtitle file (.srt)", files["subtitles"]),
            ("Father Geula character sheet", files.get("character_sheet") or "— not supplied —")]
    L = [f"# Father Geula — input bundle: {title}", "",
         f"- **Job slug:** `{slug}`",
         f"- **Deliverable:** `{slug}_flow_job.zip`, built exactly as `MASTER_PROMPT.md` specifies.",
         "",
         "## What this is",
         "`MASTER_PROMPT.md` is the complete production specification. Read all of it before "
         "planning. Every input its Part 1 lists is in this bundle, already named the way its "
         "Part 29 packaging expects:", "",
         "| Master prompt input | File in this bundle |", "|---|---|"]
    L += [f"| {a} | `{b}` |" if not b.startswith("—") else f"| {a} | {b} |" for a, b in rows]
    L += ["",
          "Copy `voiceover.mp3`, `alignment.json`, `subtitles.srt`, `voice_alignment.json`"
          + (f" and `{files['character_sheet']}`" if files.get("character_sheet") else "")
          + " into the job package byte-for-byte. `bundle_manifest.json` has a sha256 for each.",
          "",
          "## Alignment facts",
          f"- Audio duration **{payload['audio_duration_sec']:.3f}s** · speech "
          f"{va['timing_markers']['VO_SPEECH_START']:.3f}s → {va['timing_markers']['VO_SPEECH_END']:.3f}s",
          f"- Alignment QC: **{q['verdict']}**, mean confidence {q['mean_score']:.2f}",
          f"- {low} low-confidence word(s) — these are the `[?]` words of Part 6, listed in "
          "`voice_alignment.json → low_confidence_words`.",
          f"- {len(va['sentences'])} sentences · {len(va['pauses'])} pauses ≥ {PAUSE_THRESHOLD:.2f}s · "
          f"{len(va['performance_tags'])} v3 performance tag(s) placed in the timeline",
          "- Phrase anchors resolve against the words in `alignment.json` — the spoken wording "
          "with v3 tags removed — not against `elevenlabs_script.txt`.",
          f"- Script: {script_info}"]
    if q.get("flags"):
        L += ["", "### Alignment flags"] + [f"- {f}" for f in q["flags"]]
    if warnings:
        L += ["", "## Warnings"] + [f"- {w}" for w in warnings]
    return "\n".join(L) + "\n"

def build_bundle(out_dir, master_path, sheet_path, used_slugs):
    payload = json.loads((out_dir / "alignment.json").read_text(encoding="utf-8"))
    q = payload["quality"]
    title = pathlib.Path(payload["audio_file"]).stem
    slug = slugify(title)
    if slug in used_slugs:
        slug = f"{slug}_{len(used_slugs)}"
    used_slugs.add(slug)

    if q["verdict"] not in ("OK", "CHECK") and not BUNDLE_MISMATCH:
        return {"status": "skipped", "title": title,
                "why": f"verdict {q['verdict']} — fix the script and re-align first"}

    audio = find_input_file(payload["audio_file"], AUDIO_EXTS)
    if not audio:
        return {"status": "skipped", "title": title,
                "why": f"audio {payload['audio_file']} not found in in/ or in_processed/"}
    srt = out_dir / "alignment.srt"
    if not srt.exists():
        return {"status": "skipped", "title": title, "why": "alignment.srt missing — re-align"}

    details = out_dir / DETAILS_DIRNAME
    details.mkdir(exist_ok=True)
    warnings = []

    # the script as written for ElevenLabs, tags intact
    orig = details / "script_original.txt"
    script_src = orig
    if orig.exists():
        raw = orig.read_text(encoding="utf-8")
    else:
        found = find_input_file(payload.get("script_file", ""), set(SCRIPT_EXTS))
        if found:
            raw, _, _ = read_script_file(found)
            script_src = found
        elif (details / "script_used.txt").exists():
            script_src = details / "script_used.txt"
            raw = script_src.read_text(encoding="utf-8")
            warnings.append("The original script file was not found; elevenlabs_script.txt is the "
                            "cleaned narration that was aligned.")
        else:
            return {"status": "skipped", "title": title, "why": "no script found"}
    has_tags = bool(_TAG_HINT_RE.search(raw))
    script_info = ("ElevenLabs v3 script with audio tags; the tags were removed before alignment "
                   "and their positions are in voice_alignment.json → performance_tags."
                   if has_tags else
                   "plain narration — it carries no ElevenLabs v3 audio tags. Treat it as the "
                   "exact wording that was rendered.")

    va_path = details / "voice_alignment.json"
    if va_path.exists():
        va = json.loads(va_path.read_text(encoding="utf-8"))
    else:                               # job aligned before v3.6 — rebuild from alignment.json
        va = build_voice_alignment(payload, title, tags=None, tags_known=False,
                                   cleaning={"note": "aligned before v3.6; tag positions not recorded"})
        va_path.write_text(json.dumps(va, ensure_ascii=False, indent=2), encoding="utf-8")
        if has_tags:
            warnings.append("This job was aligned before v3.6, so v3 tag positions are not in "
                            "voice_alignment.json. Re-align with FORCE_REDO = True to add them.")

    zip_path  = BUNDLE_DIR / f"{slug}_input.zip"
    kick_path = BUNDLE_DIR / f"{slug}_kickoff.txt"
    sources = [out_dir / "alignment.json", srt, va_path, audio, master_path, script_src]
    if sheet_path:
        sources.append(sheet_path)
    if zip_path.exists() and kick_path.exists() and not REBUILD_BUNDLES:
        if zip_path.stat().st_mtime >= max(p.stat().st_mtime for p in sources):
            return {"status": "fresh", "title": title, "slug": slug}

    # --- gather bytes -------------------------------------------------------
    master_text, _, _ = read_script_file(master_path)
    blobs, files = {}, {}
    def put(role, name, data, media=False):
        files[role] = name
        blobs[name] = (data, media)

    put("master_prompt", "MASTER_PROMPT.md", master_text.encode("utf-8"))
    put("elevenlabs_script", "elevenlabs_script.txt", raw.encode("utf-8"))
    transcoded = audio.suffix.lower() != ".mp3"
    if transcoded:
        with tempfile.TemporaryDirectory() as td:
            mp3 = pathlib.Path(td) / "voiceover.mp3"
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(audio),
                            "-vn", "-codec:a", "libmp3lame", "-b:a", "320k", str(mp3)], check=True)
            put("voiceover", "voiceover.mp3", mp3.read_bytes(), media=True)
        warnings.append(f"The voiceover was {audio.suffix} and was converted to MP3 (320k) for the "
                        "bundle. Alignment timings were measured on the original file.")
    else:
        put("voiceover", "voiceover.mp3", audio.read_bytes(), media=True)
    put("alignment", "alignment.json", (out_dir / "alignment.json").read_bytes())
    put("subtitles", "subtitles.srt", srt.read_bytes())
    put("voice_alignment", "voice_alignment.json", va_path.read_bytes())
    if sheet_path:
        ext = sheet_path.suffix.lower().replace(".jpeg", ".jpg")
        put("character_sheet", f"character_sheet{ext}", sheet_path.read_bytes(), media=True)
    else:
        warnings.append("No character sheet was supplied. Any shot that shows Father Geula "
                        "needs it — Part 24 says to warn that the reference asset is missing.")

    kick = kickoff_message(title, slug, zip_path.name)
    manifest = {
        "bundle_schema": "father_geula_input_bundle_v1",
        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "made_by": f"Father Geula VO Alignment notebook v{NOTEBOOK_VERSION}",
        "job": {"title": title, "slug": slug, "deliverable": f"{slug}_flow_job.zip"},
        "instructions": {"read_first": "START_HERE.md", "specification": "MASTER_PROMPT.md"},
        "master_prompt": {"source_file": master_path.name,
                          "first_lines": [l.strip() for l in master_text.splitlines() if l.strip()][:2]},
        "files": {role: {"path": name, "bytes": len(blobs[name][0]),
                         "sha256": _sha256_bytes(blobs[name][0])}
                  for role, name in files.items()},
        "voiceover": {"source_file": audio.name, "transcoded_to_mp3": transcoded,
                      "duration_sec": payload["audio_duration_sec"],
                      "speech_start_sec": payload.get("speech_start_sec"),
                      "speech_end_sec": payload.get("speech_end_sec")},
        "script": {"source_file": script_src.name, "has_v3_audio_tags": has_tags,
                   "performance_tags_placed": len(va.get("performance_tags", []))},
        "alignment_quality": {"verdict": q["verdict"], "mean_score": q["mean_score"],
                              "flags": q.get("flags", []),
                              "low_confidence_words": len(va.get("low_confidence_words", []))},
        "warnings": warnings,
        "kickoff_message": kick,
    }
    start_here = start_here_md(title, slug, files, payload, va, script_info, warnings)

    # --- write zip locally, then copy to Drive in one go -----------------------
    root = f"{slug}_input/"
    with tempfile.TemporaryDirectory() as td:
        tmp = pathlib.Path(td) / zip_path.name
        with zipfile.ZipFile(tmp, "w") as z:
            z.writestr(root + "START_HERE.md", start_here.encode("utf-8"),
                       compress_type=zipfile.ZIP_DEFLATED)
            z.writestr(root + "bundle_manifest.json",
                       json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8"),
                       compress_type=zipfile.ZIP_DEFLATED)
            for name, (data, media) in blobs.items():
                z.writestr(root + name, data,
                           compress_type=zipfile.ZIP_STORED if media else zipfile.ZIP_DEFLATED)
        shutil.copyfile(tmp, zip_path)
    kick_path.write_text(kick + "\n", encoding="utf-8")
    return {"status": "built", "title": title, "slug": slug, "zip": zip_path,
            "verdict": q["verdict"], "warnings": warnings, "kickoff": kick,
            "mb": zip_path.stat().st_size / 1e6}

print("Engine loaded (v" + NOTEBOOK_VERSION + ").")


Engine loaded (v3.6).


In [ ]:
# ═══ 4) ALIGN ════════════════════════════════════════════════════════════════
import shutil, traceback, time

if not MMS_OK:
    raise RuntimeError("The MMS_FA aligner is unavailable — see the message in cell 1.")

if not JOBS:
    print("Nothing new to align — cell 5 will still bundle every finished job.")
else:
    results, last_out = [], None
    for n, job in enumerate(JOBS, start=1):
        t0 = time.time()
        print(f"\n[{n}/{len(JOBS)}] {job['audio'].name}")
        try:
            raw, kind, notes = read_script_file(job["script"])
            title = job["audio"].stem
            clean = clean_script(raw, title, STRIP_SQUARE_BRACKETS, STRIP_PARENS)

            out_dir = OUT_DIR / job["stem"]
            out_dir.mkdir(parents=True, exist_ok=True)
            # Remove legacy v3.4 root files so the folder keeps the minimal layout.
            for legacy_name in ("alignment_report.txt", "words.csv", "script_used.txt", "voice_alignment.txt"):
                legacy = out_dir / legacy_name
                if legacy.exists():
                    legacy.unlink()
            details_dir = out_dir / DETAILS_DIRNAME
            details_dir.mkdir(parents=True, exist_ok=True)
            # the script exactly as written for ElevenLabs — this is what the bundle sends
            (details_dir / "script_original.txt").write_text(raw, encoding="utf-8")

            waveform, sr = load_audio_16k(job["audio"])
            audio_seconds = waveform.shape[1] / sr
            st = _load_aligner()
            emission = compute_emission(waveform)

            # titles / ALL-CAPS lines: was it read aloud? the audio decides
            drop, decisions = resolve_ambiguous(clean, to_log_probs(emission), st["vocab"])
            text, tag_positions = assemble_script(clean, drop)
            if not text:
                raise ValueError(f"nothing spoken left after reading {job['script'].name}")
            (details_dir / "script_used.txt").write_text(text, encoding="utf-8")

            n_tags = len(tag_positions)
            job.update(script_text=text, script_kind=kind, script_notes=notes,
                       script_removals=clean["removals"], tag_positions=tag_positions,
                       ambiguous_decisions=decisions,
                       script_kind_v3="elevenlabs v3 (audio tags present)" if n_tags
                                      else "plain narration (no v3 tags found)")
            print(f"  script: {kind}, {len(text.split())} words"
                  + (f", {n_tags} v3 tag(s) set aside" if n_tags else ", no v3 tags")
                  + (f", {len(clean['removals'])} non-spoken line(s) removed" if clean["removals"] else ""))
            for note in notes:
                print("     -", note)
            for d in decisions:
                print(f"     - {d['decision']}: “{d['line'][:60]}”")

            toks, spoken_idx, norm_words = aligner_input(text, st["vocab"])
            if not norm_words:
                raise ValueError("the script has no alignable words")
            spans, num_frames = align_words(emission, norm_words)
            del emission
            release_emission()
            timings = spans_to_times(spans, num_frames, waveform.shape[1], sr)
            words = build_words(text, timings, spoken_idx, toks)

            qc = quality_check(words, audio_seconds)
            stats = write_outputs(job, words, qc, audio_seconds, st["device"], out_dir)
            last_out = out_dir

            mark = {"OK": "✓", "CHECK": "⚠️"}.get(stats["verdict"], "✗")
            print(f"  {mark} {stats['verdict']} · confidence {stats['mean_score']:.2f} · "
                  f"{stats['words']} words · {stats['sentences']} sentences · "
                  f"{stats['pauses']} pauses · {stats['suspects']} flagged · {time.time() - t0:.0f}s")
            for f in qc["flags"][:3]:
                print("     -", f)
            print(f"  → {out_dir}")
            results.append((job["audio"].name, stats["verdict"], stats["mean_score"]))

            if MOVE_WHEN_DONE:
                for f in (job["audio"], job["script"]):
                    shutil.move(str(f), str(DONE_DIR / f.name))
        except Exception:
            print("  ✗ FAILED:")
            traceback.print_exc(limit=2)

    print("\n" + "=" * 55)
    for name, verdict, score in results:
        print(f"{verdict:<16} {score:.2f}  {name}")

    if last_out:
        preview_file = last_out / DETAILS_DIRNAME / "voice_alignment.txt"
        print("\n--- preview:", f"{DETAILS_DIRNAME}/{preview_file.name}", "---\n")
        txt = preview_file.read_text(encoding="utf-8")
        print(txt[:PREVIEW_CHARS])
        if len(txt) > PREVIEW_CHARS:
            print("\n… truncated. Open the full file from Drive.")



[1/13] 23 - STRUCTURE STARVATION.mp3
  script: txt, 202 words, 14 v3 tag(s) set aside
  ⚠️ CHECK · confidence 0.94 · 198 words · 31 sentences · 33 pauses · 3 flagged · 46s
     - weak run at 0.1s: “You are not weak. You”
  → /content/drive/MyDrive/FatherGeula/VO/out/23 - structure starvation

[2/13] 24 - IS FASTING JUST SELF-HARM.mp3
  script: txt, 228 words, 13 v3 tag(s) set aside
  ⚠️ CHECK · confidence 0.92 · 226 words · 26 sentences · 33 pauses · 6 flagged · 6s
     - weak run at 0.1s: “"Isn't fasting just self-harm with”
  → /content/drive/MyDrive/FatherGeula/VO/out/24 - is fasting just self-harm

[3/13] 25 - THE TWO SOLDIERS ON THE WALL.mp3
  script: txt, 208 words, 13 v3 tag(s) set aside
  ✓ OK · confidence 0.97 · 204 words · 24 sentences · 31 pauses · 0 flagged · 6s
  → /content/drive/MyDrive/FatherGeula/VO/out/25 - the two soldiers on the wall

[4/13] 26 - A DAY UNDER THE RULE.mp3
  script: txt, 342 words, 21 v3 tag(s) set aside
  ✓ OK · confidence 0.96 · 335 words · 49 sente

In [ ]:
# ═══ 5) BUNDLES — one ready-to-send zip per finished job ═════════════════════
# Covers every job in out/, including ones aligned in earlier sessions.
# Only new or out-of-date bundles are rebuilt (REBUILD_BUNDLES = True forces all).
import traceback

if not MAKE_BUNDLES:
    print("MAKE_BUNDLES is False — skipping.")
else:
    master = find_shared(MASTER_PROMPT_NAME, MASTER_PROMPT_EXTS)
    sheet  = find_shared(CHARACTER_SHEET_NAME, CHARACTER_SHEET_EXTS)
    if not master:
        print(f"✗ No master prompt. Put {MASTER_PROMPT_NAME}.md (or .txt / .docx) in {BASE} and re-run this cell.")
    else:
        print("Master prompt  :", master.name)
        print("Character sheet:", sheet.name if sheet else "MISSING — bundles will warn")
        finished = sorted(p for p in OUT_DIR.iterdir() if (p / "alignment.json").exists())
        built, fresh, skipped, used = [], 0, [], set()
        for d in finished:
            try:
                r = build_bundle(d, master, sheet, used)
            except Exception as e:
                skipped.append((d.name, f"{type(e).__name__}: {e}"))
                traceback.print_exc(limit=1)
                continue
            if r["status"] == "built":
                built.append(r)
                mark = "✓" if r["verdict"] == "OK" else "⚠️"
                extra = f" · {len(r['warnings'])} warning(s)" if r["warnings"] else ""
                print(f"  {mark} {r['zip'].name}  ({r['mb']:.1f} MB · {r['verdict']}{extra})")
            elif r["status"] == "fresh":
                fresh += 1
            else:
                skipped.append((r["title"], r["why"]))

        print(f"\n{len(built)} built · {fresh} already up to date · {len(skipped)} skipped")
        for name, why in skipped:
            print(f"  ✗ {name}: {why}")
        print(f"\nBundles folder: {BUNDLE_DIR}")

        sample = built[-1] if built else None
        if sample:
            print("\nHOW TO SEND ONE")
            print("  1. New chat in Claude or ChatGPT (Claude: code execution must be on).")
            print("  2. Attach the zip.")
            print("  3. Paste the message from the matching _kickoff.txt, e.g.:\n")
            print("  " + sample["kickoff"].replace("\n", "\n  "))


Master prompt  : master_prompt.md
Character sheet: character_sheet.jpg
  ⚠️ 23_structure_starvation_input.zip  (1.9 MB · CHECK)
  ⚠️ 24_is_fasting_just_self_harm_input.zip  (1.8 MB · CHECK)
  ✓ 25_the_two_soldiers_on_the_wall_input.zip  (1.7 MB · OK)
  ✓ 26_a_day_under_the_rule_input.zip  (2.6 MB · OK)
  ✓ 28_hammer_1_corinthians_9_27_input.zip  (0.6 MB · OK)
  ✓ 29_can_i_pray_if_i_dont_believe_yet_input.zip  (1.6 MB · OK)
  ✓ 30_how_to_fast_without_wrecking_yourself_input.zip  (1.7 MB · OK)
  ⚠️ 31_on_being_made_input.zip  (1.7 MB · CHECK)
  ⚠️ 32_the_man_comfort_makes_input.zip  (1.8 MB · CHECK)
  ✓ 33_the_governed_man_input.zip  (1.6 MB · OK)
  ✓ 34_willpower_is_the_lie_input.zip  (1.1 MB · OK)
  ⚠️ 35_a_body_without_a_skeleton_input.zip  (1.6 MB · CHECK)
  ⚠️ 36_the_father_method_input.zip  (2.1 MB · CHECK)

13 built · 0 already up to date · 22 skipped
  ✗ 04 - WHY DO WE PRAY TO SAINTS: audio 04 - WHY DO WE PRAY TO SAINTS.mp3 not found in in/ or in_processed/
  ✗ 05 - THE MIRROR: a

---
## If something goes wrong

**The AI asks before starting, or only summarises the zip.**
Paste the kickoff message from `<slug>_kickoff.txt` — attaching the zip alone is not enough. In Claude,
code execution has to be turned on in settings for it to unzip and build files.

**"No master prompt" in cell 5.**
Cell 5 looks for `master_prompt.md` (or `.txt` / `.docx`) directly in `FatherGeula/VO`. Same idea for
`character_sheet.png / .jpg / .jpeg / .webp` — missing just adds a warning to each bundle.

**A bundle says "aligned before v3.6, tag positions not recorded".**
That job was aligned from a clean script. To get v3 tag positions into `voice_alignment.json`, put the v3
script in `in` under the same name, set `FORCE_REDO = True`, Run all.

**A title or ALL-CAPS line was kept or dropped wrongly.**
Cell 4 prints every decision, and `voice_alignment.json → script_cleaning` records it. Set
`RESOLVE_AMBIGUOUS = False` to fall back to the old rule (ALL-CAPS lines dropped, everything else kept), or
edit the script and re-run with `FORCE_REDO = True`.

**"MMS_FA aligner unavailable".**
Run `!pip install -q -U torchaudio` in a cell, restart the session, Run all.

**First run is slow.**
The aligner model is about 1.2 GB. With `CACHE_IN_DRIVE = True` it downloads once and later sessions reuse it.

**"credential propagation was unsuccessful" on a Google Doc.**
Colab's auth popup could not complete — common on mobile. Use the `.docx` download or the shared-link `.txt`.

**Verdict says MISMATCH LIKELY.**
The script is probably not what was rendered — often an edit made after the ElevenLabs render. Check
`alignment_files/script_used.txt`, fix the source, set `FORCE_REDO = True`, re-run. These jobs get no
bundle unless `BUNDLE_MISMATCH = True`; never use a MISMATCH run for automated cutting.

**Verdict says CHECK.**
One low-scoring word is usually a pronunciation quirk. A *cluster*, or a flagged weak 5-word run, means the
script and audio genuinely diverge there. CHECK jobs are bundled; their flags travel in `START_HERE.md`.
